# 07 — Relatório consolidado dos experimentos

Este notebook reúne, em um único lugar, os principais números do projeto: dados sintéticos, verdade-terreno, treinamento LTN, métricas finais e consultas compostas. É a base para `results/relatorio.md`.

In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display


def find_project_root(start: Path) -> Path:
    """Localiza a raiz do projeto subindo diretórios até achar src/ e requirements.txt."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "datasets_gerados"
RESULTS_DIR = PROJECT_ROOT / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
QUERIES_DIR = RESULTS_DIR / "queries"
TRAINING_DIR = RESULTS_DIR / "training"

print("Raiz do projeto:", PROJECT_ROOT)


Raiz do projeto: C:\Users\Sara IRMA\Documents\IA\Trabalho-Final-Raciocinio-Espacial-Neuro-Simbolico-com-LTNtorch


## 1. Dados sintéticos

In [2]:
df_objetos = pd.read_csv(DATA_DIR / "objetos_sinteticos.csv")
print(f"Total de objetos: {len(df_objetos)}")
print(df_objetos["cor"].value_counts().to_dict())
print(df_objetos["forma"].value_counts().to_dict())
print(df_objetos["tamanho"].value_counts().to_dict())

Total de objetos: 25
{'azul': 12, 'verde': 7, 'vermelho': 6}
{'triangulo': 6, 'cilindro': 6, 'circulo': 5, 'cone': 4, 'quadrado': 4}
{'grande': 13, 'pequeno': 12}


## 2. Treinamento — evolução de loss e satAgg

In [3]:
df_historico = pd.read_csv(TRAINING_DIR / "historico_treinamento.csv")
primeira, ultima = df_historico.iloc[0], df_historico.iloc[-1]

print(f"Loss total: {primeira['loss_total']:.4f} -> {ultima['loss_total']:.4f}")
print(f"satAgg:     {primeira['sat_agg']:.4f} -> {ultima['sat_agg']:.4f}")

Loss total: 1.5604 -> 1.0372
satAgg:     0.8007 -> 0.8378


## 3. Métricas finais por predicado

In [4]:
df_metricas_bin = pd.read_csv(METRICS_DIR / "metricas_finais_binarias.csv")
df_metricas_tri = pd.read_csv(METRICS_DIR / "metricas_finais_ternarias.csv")

pd.concat([df_metricas_bin, df_metricas_tri], ignore_index=True)[
    ["relacao", "accuracy", "precision", "recall", "f1"]
].sort_values("f1", ascending=False)

,relacao,accuracy,precision,recall,f1
4,closeTo,1.000000,1.000000,1.000000,1.000000
1,rightOf,0.856000,0.905660,0.786885,0.842105
0,leftOf,0.848000,0.833333,0.847458,0.840336
3,above,0.832000,0.879310,0.784615,0.829268
2,below,0.816000,0.847826,0.709091,0.772277
6,inBetween,0.722101,0.692478,0.855776,0.765515
5,canStack,0.784000,0.510638,0.857143,0.640000


## 4. Consultas compostas — respostas finais

In [5]:
pd.read_csv(QUERIES_DIR / "respostas_consultas.csv")

,id,pergunta,formula,valor_verdade,resposta
0,consulta_1,Existe algum objeto pequeno abaixo de um cilin...,"∃x(IsSmall(x) ∧ ∃y(IsCylinder(y) ∧ Below(x,y))...",0.727396,Sim
1,consulta_2,Existe um cone verde entre dois objetos?,"∃x,y,z(IsCone(x) ∧ IsGreen(x) ∧ InBetween(x,y,z))",0.839166,Sim
2,consulta_3,"Se dois triângulos estão próximos, eles têm o ...","∀x,y((IsTriangle(x) ∧ IsTriangle(y) ∧ CloseTo(...",0.926594,Sim
3,consulta_4,Existe algum objeto que pode ser empilhado sob...,"∃x,y CanStack(x,y)",0.949946,Sim


## 5. Conclusão

- Relações espaciais binárias simples (`leftOf`, `rightOf`, `below`, `above`) atingiram F1 entre 0.77 e 0.84.
- `closeTo`, tratado como predicado fuzzy determinístico baseado em distância euclidiana, atingiu F1 = 1.0.
- `canStack`, uma relação composta e desbalanceada, teve desempenho intermediário (F1 ≈ 0.64).
- `inBetween`, relação ternária, teve F1 ≈ 0.77, aceitável dada a maior dificuldade combinatória.
- O `satAgg` aumentou ao longo do treinamento (0.80 → 0.84), indicando que os axiomas lógicos foram cada vez mais satisfeitos junto com o aprendizado supervisionado.

Ver `results/relatorio.md` para a versão narrativa completa.